In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
try:
    import seaborn as sns
except:
    ! pip install seaborn
    import seaborn as sns

## Helper Classes

In [ ]:
class ExploratoryDataAnalysis:
    def __init__(self, str_uri, str_dirname_output):
        self.str_uri = str_uri
        self.str_dirname_output = str_dirname_output
        self.df_data = None

    def import_data(self):
        self.df_data = pd.read_parquet(self.str_uri)
        print(f'Data loaded: {self.df_data.shape[0]:,} rows, {self.df_data.shape[1]} columns')
        return self

    def plot_item_frequency(self, int_top_n=30):
        fig, ax = plt.subplots(figsize=(12, 8))
        ser_counts = self.df_data['item'].value_counts().head(int_top_n)
        ser_counts.plot(kind='barh', ax=ax, color='#4C72B0', edgecolor='black')
        ax.set_title(f'Top {int_top_n} Most Purchased Items', fontsize=14, y=1.02)
        ax.set_xlabel('Number of Purchases')
        ax.set_ylabel('Item')
        ax.invert_yaxis()
        plt.tight_layout()
        plt.savefig(f'{self.str_dirname_output}/01_item_frequency.png', bbox_inches='tight', dpi=150)
        plt.show()
        return self

    def plot_basket_size_distribution(self):
        ser_basket_sizes = self.df_data.groupby(
            ['member_id', 'transaction_date']
        ).size()
        fig, axes = plt.subplots(1, 2, figsize=(14, 5))
        # histogram
        axes[0].hist(ser_basket_sizes, bins=range(1, ser_basket_sizes.quantile(0.99).astype(int) + 2),
                     color='#4C72B0', edgecolor='black', alpha=0.8)
        axes[0].axvline(ser_basket_sizes.median(), color='red', linestyle='--',
                        label=f'Median: {ser_basket_sizes.median():.0f}')
        axes[0].set_title('Basket Size Distribution', fontsize=14, y=1.02)
        axes[0].set_xlabel('Items per Transaction')
        axes[0].set_ylabel('Number of Transactions')
        axes[0].legend()
        # box plot
        ser_clipped = ser_basket_sizes[ser_basket_sizes <= ser_basket_sizes.quantile(0.95)]
        axes[1].boxplot(ser_clipped, vert=True)
        axes[1].set_title('Basket Size Box Plot (95th Percentile)', fontsize=14, y=1.02)
        axes[1].set_ylabel('Items per Transaction')
        plt.tight_layout()
        plt.savefig(f'{self.str_dirname_output}/02_basket_size_distribution.png', bbox_inches='tight', dpi=150)
        plt.show()
        return self

    def plot_transactions_over_time(self):
        df_monthly = self.df_data.groupby(
            self.df_data['transaction_date'].dt.to_period('M')
        ).agg(
            int_transactions=('member_id', 'count'),
            int_unique_items=('item', 'nunique')
        ).reset_index()
        df_monthly['transaction_date'] = df_monthly['transaction_date'].dt.to_timestamp()
        fig, ax1 = plt.subplots(figsize=(14, 5))
        ax1.bar(df_monthly['transaction_date'], df_monthly['int_transactions'],
                width=25, color='#4C72B0', alpha=0.7, label='Item Purchases')
        ax1.set_xlabel('Month')
        ax1.set_ylabel('Item Purchases', color='#4C72B0')
        ax2 = ax1.twinx()
        ax2.plot(df_monthly['transaction_date'], df_monthly['int_unique_items'],
                 color='#DD8452', linewidth=2, marker='o', label='Unique Items')
        ax2.set_ylabel('Unique Items Sold', color='#DD8452')
        ax1.set_title('Transaction Volume Over Time', fontsize=14, y=1.02)
        lines1, labels1 = ax1.get_legend_handles_labels()
        lines2, labels2 = ax2.get_legend_handles_labels()
        ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper left')
        plt.tight_layout()
        plt.savefig(f'{self.str_dirname_output}/03_transactions_over_time.png', bbox_inches='tight', dpi=150)
        plt.show()
        return self

    def plot_item_frequency_distribution(self):
        ser_item_counts = self.df_data['item'].value_counts()
        fig, ax = plt.subplots(figsize=(10, 5))
        ax.hist(ser_item_counts, bins=50, color='#4C72B0', edgecolor='black', alpha=0.8)
        ax.set_title('Distribution of Item Purchase Frequencies', fontsize=14, y=1.02)
        ax.set_xlabel('Number of Times Purchased')
        ax.set_ylabel('Number of Items')
        ax.axvline(ser_item_counts.median(), color='red', linestyle='--',
                    label=f'Median: {ser_item_counts.median():.0f}')
        ax.set_yscale('log')
        ax.legend()
        plt.tight_layout()
        plt.savefig(f'{self.str_dirname_output}/04_item_frequency_distribution.png', bbox_inches='tight', dpi=150)
        plt.show()
        return self

    def plot_member_activity(self):
        ser_member_transactions = self.df_data.groupby('member_id')['transaction_date'].nunique()
        fig, ax = plt.subplots(figsize=(10, 5))
        ax.hist(ser_member_transactions, bins=50, color='#4C72B0', edgecolor='black', alpha=0.8)
        ax.set_title('Member Activity Distribution', fontsize=14, y=1.02)
        ax.set_xlabel('Number of Transactions per Member')
        ax.set_ylabel('Number of Members')
        ax.axvline(ser_member_transactions.median(), color='red', linestyle='--',
                    label=f'Median: {ser_member_transactions.median():.0f}')
        ax.legend()
        plt.tight_layout()
        plt.savefig(f'{self.str_dirname_output}/05_member_activity.png', bbox_inches='tight', dpi=150)
        plt.show()
        return self

    def plot_day_of_week(self):
        self.df_data['day_of_week'] = self.df_data['transaction_date'].dt.day_name()
        list_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
        ser_dow = self.df_data['day_of_week'].value_counts().reindex(list_order)
        fig, ax = plt.subplots(figsize=(10, 5))
        ser_dow.plot(kind='bar', ax=ax, color='#4C72B0', edgecolor='black')
        ax.set_title('Purchases by Day of Week', fontsize=14, y=1.02)
        ax.set_xlabel('Day of Week')
        ax.set_ylabel('Number of Item Purchases')
        ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right')
        for i, int_val in enumerate(ser_dow.values):
            ax.text(i, int_val + 50, f'{int_val:,}', ha='center', fontsize=10)
        plt.tight_layout()
        plt.savefig(f'{self.str_dirname_output}/06_day_of_week.png', bbox_inches='tight', dpi=150)
        plt.show()
        self.df_data = self.df_data.drop(columns=['day_of_week'])
        return self

    def print_summary_table(self):
        print('\n=== DATASET SUMMARY ===')
        print(f'Total rows (item-level): {len(self.df_data):,}')
        print(f'Unique members: {self.df_data["member_id"].nunique():,}')
        print(f'Unique items: {self.df_data["item"].nunique():,}')
        int_transactions = self.df_data.groupby(['member_id', 'transaction_date']).ngroups
        print(f'Total transactions: {int_transactions:,}')
        flt_avg_basket = len(self.df_data) / int_transactions
        print(f'Avg items per transaction: {flt_avg_basket:.2f}')
        print(f'Date range: {self.df_data["transaction_date"].min().date()} to {self.df_data["transaction_date"].max().date()}')
        return self

## Constants

In [ ]:
str_bucket = 'market-basket-analysis-demo'
str_task = 'eda'
str_dirname_output = './output'
str_uri = f's3://{str_bucket}/00_data_collection/groceries.parquet'

## Output Directory

In [ ]:
try:
    os.mkdir(str_dirname_output)
except FileExistsError:
    pass
print(f'Output directory ready: {str_dirname_output}')

## Run EDA

In [ ]:
cls_eda = ExploratoryDataAnalysis(str_uri, str_dirname_output)
cls_eda.import_data()

In [ ]:
cls_eda.plot_item_frequency()

In [ ]:
cls_eda.plot_basket_size_distribution()

In [ ]:
cls_eda.plot_transactions_over_time()

In [ ]:
cls_eda.plot_item_frequency_distribution()

In [ ]:
cls_eda.plot_member_activity()

In [ ]:
cls_eda.plot_day_of_week()

In [ ]:
cls_eda.print_summary_table()

## Completion

In [ ]:
print('\n=== EDA COMPLETE ===')
print(f'Visualizations saved to: {str_dirname_output}')